[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safe-t-ai/safe-t-ai.github.io/blob/main/notebooks/02_test1_volume_estimation.ipynb)

# 02 — Test 1: Volume Estimation Bias

Generates simulated AI volume predictions with documented demographic bias and produces
all frontend JSON files for the Test 1 (Volume Estimation) tab.

**Prerequisite:** `01_fetch_data.ipynb` must have run successfully so that
`backend/data/raw/durham_census_tracts.geojson` exists.

Outputs written:
- `backend/data/simulated/ground_truth_counters.json`
- `backend/data/simulated/ai_volume_predictions.json`
- `backend/data/simulated/tract_volume_predictions.json`
- `frontend/public/data/census-tracts.json`
- `frontend/public/data/counter-locations.json`
- `frontend/public/data/volume-report.json`
- `frontend/public/data/choropleth-data.json`
- `frontend/public/data/accuracy-by-income.json`
- `frontend/public/data/accuracy-by-race.json`
- `frontend/public/data/scatter-data.json`

In [1]:
# Install dependencies
%pip install -q geopandas requests shapely numpy pandas

In [2]:
# Bootstrap repo — clones if absent, adds backend/ to sys.path
import subprocess, sys
from pathlib import Path

NOTEBOOKS_URL = "https://raw.githubusercontent.com/safe-t-ai/safe-t-ai.github.io/main/notebooks/colab_utils.py"
utils_path = Path("/content/colab_utils.py")
if not utils_path.exists():
    import urllib.request
    urllib.request.urlretrieve(NOTEBOOKS_URL, str(utils_path))

sys.path.insert(0, "/content")
import colab_utils

repo = colab_utils.prepare_notebook(pull_latest=True)
print(f"Repo ready at {repo}")

Repo ready at /content/safe-t-ai.github.io


## 1. Load Census Data

In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
from config import RAW_DATA_DIR, SIMULATED_DATA_DIR, BIAS_PARAMETERS, VOLUME_SIMULATION_CONFIG, CENSUS_VINTAGE
from utils.demographic_analysis import calculate_income_quintiles

SIMULATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

census_file = RAW_DATA_DIR / 'durham_census_tracts.geojson'
if not census_file.exists():
    raise FileNotFoundError(
        f"Census data not found at {census_file}.\n"
        "Run 01_fetch_data.ipynb first."
    )

census_gdf = gpd.read_file(census_file)
print(f"Loaded {len(census_gdf)} census tracts")

Loaded 67 census tracts


## 2. Simulate Ground Truth Counters

In [4]:
def generate_ground_truth_counters(census_gdf):
    """Generate ground truth bike/ped counter data anchored to census tracts."""
    num_counters = VOLUME_SIMULATION_CONFIG['num_counters']
    counters = []
    for idx in range(num_counters):
        tract = census_gdf.iloc[idx % len(census_gdf)]
        centroid = tract.geometry.centroid
        base_volume = tract['total_population'] / 100
        seasonal_factor = np.random.uniform(0.8, 1.2)
        daily_volume = int(base_volume * seasonal_factor)
        counters.append({
            'counter_id': f'CTR{idx+1:03d}',
            'tract_id': tract['tract_id'],
            'lat': centroid.y,
            'lon': centroid.x,
            'daily_volume': daily_volume,
            'median_income': tract['median_income'],
            'pct_minority': tract['pct_minority'],
            'type': 'real' if idx < 3 else 'simulated',
        })
    df = pd.DataFrame(counters)
    out = SIMULATED_DATA_DIR / 'ground_truth_counters.json'
    df.to_json(out, orient='records', indent=2)
    print(f"Generated {len(counters)} counter locations -> {out}")
    return df


ground_truth = generate_ground_truth_counters(census_gdf)

Generated 15 counter locations -> /content/safe-t-ai.github.io/backend/data/simulated/ground_truth_counters.json


## 3. Apply AI Bias to Counter Predictions

In [5]:
def calculate_demographic_bias(income_quintile, pct_minority):
    """Combined income + racial bias multiplier from config parameters."""
    if income_quintile <= 2:
        income_bias = 1.0 - BIAS_PARAMETERS['low_income_undercount']
    elif income_quintile >= 4:
        income_bias = 1.0 + BIAS_PARAMETERS['high_income_overcount']
    else:
        income_bias = 1.0

    if pct_minority > VOLUME_SIMULATION_CONFIG['minority_high_threshold']:
        racial_bias = 1.0 - BIAS_PARAMETERS['minority_undercount']
    elif pct_minority < VOLUME_SIMULATION_CONFIG['minority_low_threshold']:
        racial_bias = 1.0 + VOLUME_SIMULATION_CONFIG['minority_low_overcount']
    else:
        racial_bias = 1.0

    return income_bias * racial_bias


def apply_ai_bias(ground_truth_df, census_gdf):
    """Apply documented bias patterns to counter-level ground truth."""
    census_quintiled = calculate_income_quintiles(
        census_gdf[['tract_id', 'median_income']].copy()
    )
    df = ground_truth_df.merge(
        census_quintiled[['tract_id', 'income_quintile']],
        on='tract_id', how='left',
    )
    predictions = []
    for _, counter in df.iterrows():
        true_volume = counter['daily_volume']
        total_bias = calculate_demographic_bias(counter['income_quintile'], counter['pct_minority'])
        noise = np.random.normal(1.0, BIAS_PARAMETERS['base_noise'])
        predicted = int(true_volume * total_bias * noise)
        predictions.append({
            'counter_id': counter['counter_id'],
            'tract_id': counter['tract_id'],
            'true_volume': true_volume,
            'predicted_volume': predicted,
            'error': predicted - true_volume,
            'error_pct': (predicted - true_volume) / true_volume * 100,
            'income_quintile': counter['income_quintile'],
            'pct_minority': counter['pct_minority'],
            'bias_applied': total_bias,
        })
    result = pd.DataFrame(predictions)
    out = SIMULATED_DATA_DIR / 'ai_volume_predictions.json'
    result.to_json(out, orient='records', indent=2)
    print(f"Saved AI predictions to {out}")
    low = result[result['income_quintile'] <= 2]
    high = result[result['income_quintile'] >= 4]
    print(f"  Low-income mean error:  {low['error_pct'].mean():+.1f}%")
    print(f"  High-income mean error: {high['error_pct'].mean():+.1f}%")
    return result


ai_predictions = apply_ai_bias(ground_truth, census_gdf)

Saved AI predictions to /content/safe-t-ai.github.io/backend/data/simulated/ai_volume_predictions.json
  Low-income mean error:  -36.0%
  High-income mean error: +7.9%


## 4. Generate Tract-Level Predictions

In [6]:
def generate_tract_level_predictions(census_gdf):
    """Simulate AI volume predictions for all census tracts."""
    tract_summary = census_gdf.groupby('tract_id').agg({
        'total_population': 'sum',
        'median_income': 'first',
        'pct_minority': 'first',
        'geometry': 'first',
    }).reset_index()
    tract_summary = calculate_income_quintiles(tract_summary)

    base_rate = VOLUME_SIMULATION_CONFIG['base_active_transport_rate']
    predictions = []
    for _, tract in tract_summary.iterrows():
        population = tract['total_population']
        area_km2 = tract.geometry.area * 111 * 111
        density = population / area_km2 if area_km2 > 0 else 0

        density_factor = VOLUME_SIMULATION_CONFIG['density_default_factor']
        for threshold, factor in VOLUME_SIMULATION_CONFIG['density_thresholds']:
            if density > threshold:
                density_factor = factor
                break

        true_vol = int(population * base_rate * density_factor)
        total_bias = calculate_demographic_bias(tract['income_quintile'], tract['pct_minority'])
        noise = np.random.normal(1.0, VOLUME_SIMULATION_CONFIG['aggregate_noise_std'])
        predicted_vol = int(true_vol * total_bias * noise)
        error = predicted_vol - true_vol
        error_pct = (error / true_vol * 100) if true_vol > 0 else 0

        predictions.append({
            'tract_id': tract['tract_id'],
            'true_volume': true_vol,
            'predicted_volume': predicted_vol,
            'error': error,
            'error_pct': error_pct,
            'income_quintile': tract['income_quintile'],
            'median_income': tract['median_income'],
            'pct_minority': tract['pct_minority'],
            'total_population': population,
            'bias_applied': total_bias,
        })

    result = pd.DataFrame(predictions)
    out = SIMULATED_DATA_DIR / 'tract_volume_predictions.json'
    result.to_json(out, orient='records', indent=2)
    print(f"Saved {len(result)} tract predictions to {out}")
    print(f"  Overall bias: {result['error_pct'].mean():+.1f}%")
    return result


tract_predictions = generate_tract_level_predictions(census_gdf)

Saved 67 tract predictions to /content/safe-t-ai.github.io/backend/data/simulated/tract_volume_predictions.json
  Overall bias: -14.7%


## 5. Generate Frontend JSON — Test 1

In [7]:
import json
from pathlib import Path
from models.volume_estimator import VolumeEstimationAuditor, load_test1_data
from utils.geospatial import geojson_to_dict, simplify_geometry
from utils.demographic_analysis import calculate_income_quintiles, calculate_minority_category

output_dir = repo / 'frontend' / 'public' / 'data'
output_dir.mkdir(parents=True, exist_ok=True)

# Reload from files so auditor uses consistent serialised state
census_gdf_loaded, ground_truth_loaded, ai_predictions_loaded = load_test1_data(
    RAW_DATA_DIR, SIMULATED_DATA_DIR
)
auditor = VolumeEstimationAuditor(census_gdf_loaded, ground_truth_loaded, ai_predictions_loaded)
print("Auditor ready.")

Auditor ready.


In [8]:
# volume-report.json
print("Generating volume-report.json...")
report = auditor.generate_full_report()
report["_provenance"] = {
    "data_type": "mixed",
    "real": [
        f"US Census ACS {CENSUS_VINTAGE} demographics (tract-level income, minority %)",
        "Durham census tract geometries (Census TIGER)",
    ],
    "simulated": [
        "Counter locations (15 randomly placed, income-stratified)",
        "Ground truth pedestrian/cyclist volumes (income-correlated synthetic)",
        "AI volume predictions (bias parameters are simulation assumptions, not empirically measured values)",
    ],
    "note": (
        "Test 1 is a methodology demonstration. Bias parameters (low_income_undercount=0.25, "
        "minority_undercount=0.20) are plausible simulation assumptions. Their direction — that "
        "crowdsourced active transportation data underrepresents low-income and minority populations "
        "— is supported by peer-reviewed research: Roy et al. (2019) found a ~12% increase in "
        "Strava counts per 10% increase in white population share (doi:10.3390/urbansci3020062); "
        "Williams & Behrendt (2025) document demographic bias and social justice implications of "
        "Strava Metro data in transport planning (doi:10.1080/17450101.2025.2542163). "
        "The specific magnitudes are not empirically derived. No real vendor AI predictions are included."
    ),
}
with open(output_dir / 'volume-report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("Done.")

Generating volume-report.json...
Done.


In [9]:
# choropleth-data.json
print("Generating choropleth-data.json...")
tract_preds_df = pd.read_json(SIMULATED_DATA_DIR / 'tract_volume_predictions.json')
tract_preds_df['tract_id'] = tract_preds_df['tract_id'].astype(str)
census_gdf_loaded['tract_id'] = census_gdf_loaded['tract_id'].astype(str)

tract_geoms = census_gdf_loaded.dissolve(by='tract_id', aggfunc='first')[['geometry']].reset_index()
tract_errors_gdf = tract_geoms.merge(
    tract_preds_df[['tract_id', 'error_pct', 'error', 'true_volume', 'predicted_volume',
                    'median_income', 'pct_minority', 'total_population']],
    on='tract_id', how='left',
)
tract_errors_gdf = calculate_income_quintiles(tract_errors_gdf)
tract_errors_gdf = calculate_minority_category(tract_errors_gdf)
tract_errors_gdf = simplify_geometry(tract_errors_gdf, tolerance=0.001)

with open(output_dir / 'choropleth-data.json', 'w') as f:
    json.dump(geojson_to_dict(tract_errors_gdf), f)
print(f"  {len(tract_errors_gdf)} tracts written.")

Generating choropleth-data.json...
  67 tracts written.


In [10]:
# accuracy-by-income.json, accuracy-by-race.json, scatter-data.json
print("Generating accuracy-by-income.json...")
with open(output_dir / 'accuracy-by-income.json', 'w') as f:
    json.dump(auditor.analyze_by_income(), f, indent=2)

print("Generating accuracy-by-race.json...")
with open(output_dir / 'accuracy-by-race.json', 'w') as f:
    json.dump(auditor.analyze_by_race(), f, indent=2)

print("Generating scatter-data.json...")
with open(output_dir / 'scatter-data.json', 'w') as f:
    json.dump(auditor.get_scatter_data(), f, indent=2)

print("All Test 1 frontend files written.")

Generating accuracy-by-income.json...
Generating accuracy-by-race.json...
Generating scatter-data.json...
All Test 1 frontend files written.


## Publish Artifacts

In [ ]:
notebook_path = colab_utils.save_notebook("02_test1_volume_estimation.ipynb", repo_dir=repo)

paths = [
    "backend/data/simulated/ground_truth_counters.json",
    "backend/data/simulated/ai_volume_predictions.json",
    "backend/data/simulated/tract_volume_predictions.json",
    "frontend/public/data/census-tracts.json",
    "frontend/public/data/counter-locations.json",
    "frontend/public/data/volume-report.json",
    "frontend/public/data/choropleth-data.json",
    "frontend/public/data/accuracy-by-income.json",
    "frontend/public/data/accuracy-by-race.json",
    "frontend/public/data/scatter-data.json",
]
if notebook_path:
    paths.insert(0, notebook_path)

colab_utils.publish_artifacts(
    paths=paths,
    message="data: regenerate Test 1 volume estimation",
    repo_dir=repo,
)